# Week 3: MPOs

In [ ]:
using Pkg
Pkg.activate("../")
using ITensors
using ITensorMPS
using Plots
using LinearAlgebra

## Exercise 1

Let us consider the XY model (with $\gamma=0$) discussed in the `Extended example` in the companion lecture-notebook `MPO_ITensor.ipynb`. 
This time we add a constant magnetic field in the $Z$ direction. The new Hamiltonian reads:
$$
H = -J \sum_{i=1}^{N-1} \sigma_-^{i} \sigma_+^{i+1} + \sigma_+^{i} \sigma_-^{i+1} -h \sum_{i=1}^N \sigma_z^i
$$ 
with $N=20$, $J = 1.0$ and $h=0.5$($\hbar = 1$).

 and the initial state $\ket{\psi_0} = \ket{\uparrow \downarrow \downarrow \ldots \downarrow}$ where only the first spin is "up" and all the others are down. Since $[H,N_z$] = 0$, $N_z$, i.e. the number of spins up is a constant of the motion. Thus $\left \{\ket{i} \right \}_{i=1}^N$ is a basis for the subspace of the Hilbert space of the system actually visited during the evolution.

1. Form the tridiagonal matrix representing $H$ in the $\ket{i}$ basis (see [LinearAlgebra documentation](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.Tridiagonal)).
2. Determine the solution of the eigensystem and the solutions $\psi(j,t)$ for the amplitudes of being at site $j$ at time $t$ as a function

`solExactHom(nn::Int, jj:Float, h::Float, t::Float)::Vector{ComplexF64}`

that takes the number of sites $N$, the value of the coupling constant $J$, the intensity of the magnetic field $h$ and the time $t$ and returns a vector of $N$ complex values $[\psi(1,t),\psi(2,t),\ldots,\psi(N,t)]$ with $\psi(j,t)$ the amplitude of the basis state $\ket{j}$ at time $t$.

3. Determine the amplitudes $[\psi(1,t),\psi(2,t),\ldots,\psi(N,t)]$ for $t=0, 0.1,0.2,\ldots,10$. 

Question: do we really need to compute the amplitudes? How does this Hamiltonian relate to the Hamiltonian used the `Extended example` of the companion lecture-notebook `MPO_ITensor.ipynb`?


In [ ]:
#Construct the Sub Hamiltonian

J = 1.0
N = 20
h = 0.5
dl = -J*ones(ComplexF64, 19)

In [ ]:
d = -h*(N-2)*ones(ComplexF64, 20) # N-1 down, 1 up

In [ ]:
du = dl

In [ ]:
Tri = Tridiagonal(dl, d, du)

In [ ]:
# diag

eigenval, eigenvek = eigen(Tri)

In [ ]:
# Calculate the time dept versions of the vectors

function solExactHam(nn::Int, jj::Float64, h::Float64, t::Float64)::Vector{ComplexF64}
    J = jj
    N = nn
    h = h
    dl = -J*ones(ComplexF64, nn-1)

    d = -h*(N-2)*ones(ComplexF64, nn) # N-1 down, 1 up
    du = dl
    Tri = Tridiagonal(dl, d, du)
    eigenval, eigenvek = eigen(Tri)

    # Initial state: (up, dn, dn, ...) -> (1, 0, 0) -> Projection on an eigenstate is the first component of that state.
    coeffs = eigenvek[1, :]
    #time evolution
    for l in range(1, N)
        coeffs[l] = coeffs[l]*exp( - 1im*t*eigenval[l])
    end

    # construct back to the state

    return eigenvek*coeffs
end
    
    

In [ ]:
Psi_0 = zeros(N)
Psi_0[1] = 1

Psi_t = solExactHam(20, 1., 0.5, 1.)
norm(Psi_t)

In [ ]:
timeStates = []
for te in 0:0.1:10
    push!(timeStates, solExactHam(20, 1., 0.5, te))
end
timeStates

## Exercise 2

Consider the Hamiltonian $H$ of the previous exercise. 

1. Design the bulk matrix and the bounday vectors producing the MPO (see Section 3.2.2 of the lecture notes) representation of $H$.
2. Determine the new MPO via `OpSum` and `MPO` constructs.
3. Check the bond dimension of the MPO: is it the same as the bond dimension in the absence of the $\sigma_z$ term (see Extended example)? Yes/no,why? How does the ITensor MPO bond dimension compare to the bond dimension of the bulk matrix/vectors determined at point 1?

In [ ]:
# Define state in ITensorMPS

nn = 20
jj=1.0
system = siteinds("S=1/2",nn)
#Prepare the initial state as a product state with "Up" on the first site and "Dn" on the rest of the sites
psi0= MPS(ComplexF64,system,[(j==1) ? "Up" : "Dn" for j in 1:nn]);
#Set the orthogonality center to the first site
orthogonalize!(psi0,1)



In [ ]:
#define the hamiltonian

h = -0.5
opsum = OpSum()
for bond in 1:nn-1
    #note that the sintax allows for constants to appear before the operator string
    opsum += -1. * jj, "S-", bond, "S+", bond+1
    opsum += -1. * jj, "S+", bond, "S-", bond+1
end
for l in 1:nn
    opsum += h, "Z", l
end

ham = MPO(opsum, system)

-> Bond dimension is now 4


Plots und Zeitevolution:

In [ ]:
deltat = 0.1
nsteps = Int(round(10/deltat))
hdeltat = deltat * ham
#build an MPO with the operator "Id" on each site
theOne = MPO(system,"Id")
factor(k) = (-1im)^k / factorial(k)
#As not to have all the indices of hdeltat contracted with each other, we prime the second one and then we contract the first index of the second one with the first index of the first one, and then we lower the second prime level to first prime level.
h2 = mapprime(prime(hdeltat)*hdeltat,2 =>1)
h3 = mapprime(prime(hdeltat) * h2,2 =>1)
terms = [theOne, factor(1)*hdeltat, factor(2)*h2, factor(3)*h3];
appo = terms[1];
for t in terms[2:end]
    appo = t + appo
end
h4thOrder = appo;
#h4thOrder = +(terms...; cutoff=1e-18);

In [ ]:
evolved = Vector{MPS}()
push!(evolved, psi0)
for j=2:nsteps
    push!(evolved, noprime(h4thOrder * evolved[end]))
end

In [ ]:
function measureProjUp(psi::MPS,j::Int)
    #Measured site must be the orthogonality center
    orthogonalize!(psi,j)
    return scalar(dag(psi[j])*noprime(op("ProjUp", system[j])*psi[j]))
end

In [ ]:
function measureZ(psi::MPS,j::Int)
    #Measured site must be the orthogonality center
    orthogonalize!(psi,j)
    return scalar(dag(psi[j])*noprime(op("Z", system[j])*psi[j]))
end

In [ ]:
anim = @animate for i=1:nsteps
    fig = plot(real([measureProjUp(evolved[i],j) for j in 1:nn]), seriestype=:scatter, label="ProjUp")
    plot!(fig, real([measureProjUp(evolved[i],j) for j in 1:nn]))
end;
gif(anim, "evolution.gif", fps=10)

In [ ]:
anim = @animate for i=1:nsteps
    fig = plot(real([measureZ(evolved[i],j) for j in 1:nn]), seriestype=:scatter, label="measureZ")
    plot!(fig, real([measureZ(evolved[i],j) for j in 1:nn]))
end;
gif(anim, "evolution.gif", fps=10)

## Exercise 3

The file `onSiteEnergies.dat` contains 20 real numbers representing the values of the $h_i$ terms apparing in the Hamiltonian:

$$
H = -J \sum_{i=1}^{N-1} \sigma_-^{i} \sigma_+^{i+1} + \sigma_+^{i} \sigma_-^{i+1} -\sum_{i=1}^N h_i \sigma_z^i
$$ 

You can read the file by using, for example, the library `DelimitedFiles` and executing the command:
`readdlm("onSiteEnergies.dat")` from within this folder, or by providing the relative/absolute path to the file. Though in the presence of an inhomogeneous magnetic field the relation $[H,N_z]$ still holds.

1. Determine, by numerical means, the eigensystem of the Hamiltonian in the single excitation subspace, that is the space spanned by the _position_ basis $\left \{\ket{i} \right \}_{i=1}^{20}$, where the state $\ket{i}$ has the spin up at site $i$. This is again a tridiagonal matrix...
2. Determine the numerically exact evolution of the  initial state $\ket{\psi_0} = \ket{1}$ using the result of the previous point. Record the probabilities $|\braket{j|\psi(t)}|^2,\; j=1,2,\ldots,20$ for $t=0,0.1,0.2,\ldots,10$.
3. Determine the MPO corresponding to the same Hamiltonian, the MPO corresponding to e 4-th order expansion of $e^{-i H t}$, as done in the companion lecture-notebook.
4. Compare the ``exact'' and the MPO-MPS solutions by assesing the probabilities of point 2, as determined by the MPS-MPO evolution, at the same times as in point 2.

Physically speaking: What do you observe? What is happening?

In [ ]:
# read the file
using DelimitedFiles
magn = vec(readdlm("onSiteEnergies.dat"))

In [ ]:
#For the exact time vol: Modify funktion from ex. 1
function solExactHam(nn::Int, jj::Float64, h::Vector{Float64}, t::Float64)::Vector{ComplexF64}
    J = jj
    N = nn
    h = h
    dl = -J*ones(ComplexF64, nn-1)

    d = zeros(ComplexF64, N) 
    for l in 1:N
        d[l] = (-1.)*sum(h) + (2.)*h[l]
    end
    du = dl
    Tri = Tridiagonal(dl, d, du)
    eigenval, eigenvek = eigen(Tri)

    # Initial state: (up, dn, dn, ...) -> (1, 0, 0) -> Projection on an eigenstate is the first component of that state.
    coeffs = eigenvek[1, :]
    #time evolution
    for l in range(1, N)
        coeffs[l] = coeffs[l]*exp( - 1im*t*eigenval[l])
    end

    # construct back to the state

    return eigenvek*coeffs
end

In [ ]:
exactStates = []
for te in 0:0.01:10
    push!(exactStates, solExactHam(20, 1., magn, te))
end
exactStates

In [ ]:
normCoorect = true
for vec in exactStates
    if !( norm(vec) ≈ 1)
        normCoorect = false
    end
end
normCoorect


In [ ]:
# MPO Implement
opsum = OpSum()
for bond in 1:nn-1
    #note that the sintax allows for constants to appear before the operator string
    opsum += -1. * jj, "S-", bond, "S+", bond+1
    opsum += -1. * jj, "S+", bond, "S-", bond+1
end
for l in 1:nn
    opsum += magn[l], "Z", l
end

ham = MPO(opsum, system)

In [ ]:
deltat = 0.01
nsteps = Int(round(10/deltat))
hdeltat = deltat * ham
#build an MPO with the operator "Id" on each site
theOne = MPO(system,"Id")
factor(k) = (-1im)^k / factorial(k)
#As not to have all the indices of hdeltat contracted with each other, we prime the second one and then we contract the first index of the second one with the first index of the first one, and then we lower the second prime level to first prime level.
h2 = mapprime(prime(hdeltat)*hdeltat,2 =>1)
h3 = mapprime(prime(hdeltat) * h2,2 =>1)
terms = [theOne, factor(1)*hdeltat, factor(2)*h2, factor(3)*h3];
appo = terms[1];
for t in terms[2:end]
    appo = t + appo
end
h4thOrder = appo;
#h4thOrder = +(terms...; cutoff=1e-18);

In [ ]:
evolved = Vector{MPS}()
push!(evolved, psi0)
for j=2:nsteps
    push!(evolved, noprime(h4thOrder * evolved[end]))
end

evolved[1]

In [ ]:
# obtain from the MPS the amplitude Psi(j, t) to the respective substate.

# Explicit Representation in the upper space is large

# Construct the substates as MPS's, and calculate the projection to the evolved ones. 

basisJ = []
for l in 1:N
    push!(basisJ, MPS(ComplexF64,system,[(j==l) ? "Up" : "Dn" for j in 1:nn]))
end

In [ ]:
amplitudes = []
for evol in evolved
    amp = zeros(ComplexF64, N)
    for l in 1:N
        amp[l] = inner(basisJ[l], evol)
    end
    push!(amplitudes, amp)
end

In [ ]:
amplitudes[400]

In [ ]:
exactStates[400]

## Exercise 4

Consider the Hamiltonian

$$
H = -J \sum_{i=1}^{N-1} \sigma_-^{i} \sigma_+^{i+1} + \sigma_+^{i} \sigma_-^{i+1} 
$$ 

The initial state is this time the state 
$$
\ket{\psi_0} = \sigma_+^1 \sigma_+^2 \ket{0},
$$
where $\ket{0}$ indicates the state with all the spin down (no excitations, if you want). The new state has two spins up at sites 1 and 2 and the remaining down. Since $H$ conserves the number of spin up, the evolution lives in the space $H_{|N=2}$.
Since we have the analytic solution for the single excitation space, 
$$
\ket{\phi_k} = \sum_{j=1}^N \sqrt{\frac{2}{N+1}} \sin \left (\frac{\pi k j}{N+1}\right ) \ket{j}
$$

$$
\epsilon_k = -2 J \cos \left( \frac{k \pi}{N+1} \right )
$$
we can detrmine the solution of the two excitation case 
$$
\ket{\psi(t)} = \sum_{j_1 < j_2} \psi_{j_1,j_2}(t) \ket{j_1,j_2}
$$
by means of Slater determinant:

$$
\psi_{j_1,j_2}(t) = \frac{1}{\sqrt{2}} \det 
\begin{pmatrix}
S(j_1,j_1^{0},t) & S(j_1,j_2^{0},t) \\
 S(j_2,j_1^{0},t) & S(j_2,j_2^{0},t)
\end{pmatrix}
$$
where $S(j_m,j_n^0,t)$ are the single particle amplitudes at position $j_m$ when the excitation is at site $j_n^0$ at time $t=0$, i.e.
$$
S(j_m,j_n^0,t) = \sum_{k=1}^N e^{-i t \epsilon_k} \frac{2}{N+1} \sin \left (\frac{\pi k j_m}{N+1}\right ) \sin \left (\frac{\pi k j_n^0}{N+1}\right ).
$$

1. Determine the probabilities $|\psi_{j_1,j_2}(t)|^2\; j_1,j_2 = 1,2,...,\ldots,20$ for $t=0,0.1,0.2,\ldots,10$ using the exact, Slater-determinant based, solution for the same time-steps as in the previous exercise starting from $\ket{\psi_0}$.
2. By using the MPO defined in the companion notebook, and the corresponding $4$-th order expansion of $e^{-i H t}$, determine the evolution of the same initial state and the probabilities $|\psi_{j_1,j_2}(t)|^2\; j_1,j_2 = 1,2,...,\ldots,20$ for $t=0,0.1,0.2,\ldots,10$.
3. Compare, either numerically or graphically, the occupation probabilities obtained at point 1. and 2..

In [ ]:
# analytische definitionen. 
function eps(k)
    return -2*J*cos(k*pi/(N+1))
end

function S(j_m, j_n_0, t)
    res = 0
    for k in 1:N
        res += exp(-1*1im*eps(k)*t)*2/(N+1)*sin(pi*k*j_m/(N+1))*sin(pi*k*j_n_0/(N+1))
    end
    return res
end
S(1, 2, 3)



In [ ]:
function Psi_1_2(j_1, j_2, t)
    return (S(j_1, 1, t)*S(j_2, 2, t) - S(j_1, 2, t)*S(j_2, 1, t)) # 1/sqrt 2 weg !!!
end
Psi_1_2(1, 2, 3) 

In [ ]:
# Obtain the same amplitudes from the MPS 

nn = 20
jj=1.0
system = siteinds("S=1/2",nn)
psi0= MPS(ComplexF64,system,[(j==1 || j==2) ? "Up" : "Dn" for j in 1:nn]);
#Set the orthogonality center to the first site
orthogonalize!(psi0,1)

In [ ]:
opsum = OpSum()
for bond in 1:nn-1
    #note that the sintax allows for constants to appear before the operator string
    opsum += -1. * jj, "S-", bond, "S+", bond+1
    opsum += -1. * jj, "S+", bond, "S-", bond+1
end
ham = MPO(opsum, system)

In [ ]:
deltat = 0.01
nsteps = Int(round(10/deltat))
hdeltat = deltat * ham
#build an MPO with the operator "Id" on each site
theOne = MPO(system,"Id")
factor(k) = (-1im)^k / factorial(k)
#As not to have all the indices of hdeltat contracted with each other, we prime the second one and then we contract the first index of the second one with the first index of the first one, and then we lower the second prime level to first prime level.
h2 = mapprime(prime(hdeltat)*hdeltat,2 =>1)
h3 = mapprime(prime(hdeltat) * h2,2 =>1)
terms = [theOne, factor(1)*hdeltat, factor(2)*h2, factor(3)*h3];
appo = terms[1];
for t in terms[2:end]
    appo = t + appo
end
h4thOrder = appo;
#h4thOrder = +(terms...; cutoff=1e-18);

In [ ]:
evolved = Vector{MPS}()
push!(evolved, psi0)
for j=2:nsteps
    push!(evolved, noprime(h4thOrder * evolved[end]))
end

evolved[1]

In [ ]:
function Amp(j_1, j_2, t)
    MPS_t = evolved[Int(round(t/deltat))+1]
    return inner(MPS_t, MPS(ComplexF64,system,[(j==j_1 || j==j_2) ? "Up" : "Dn" for j in 1:nn]))
end

In [ ]:
Amp(1, 2, 3)

In [ ]:
Psi_1_2(1, 2, 0)

In [ ]:
times = 0:deltat:9.9

plot(times, norm.(Psi_1_2.(5, 6, times)))
plot!(times, norm.(Amp.(5, 6, times)))

## Exercize 5

Suppose you want to the evolution an initial state of the same system of $N=20$ spins where, this time, $N_{up}=10$ spins are initially up (and the remaining down). The spins up can be placed where you like. Is it possible to determine the evolution using the exact solution (with Slater determinant generalized to $M=10$)? How complex is it?

1. Prepare the MPS representing an initial state with 10 spin up at the beginning of the chain.
2. Prepare (copy) the MPO for the Hamiltonian

$$
H = -J \sum_{i=1}^{N-1} \sigma_-^{i} \sigma_+^{i+1} + \sigma_+^{i} \sigma_-^{i+1} 
$$ 
for $J=1.0$ and $N=20$.

3. Determine the MPO for the 4-th order expansion of  $e^{-itH}$.

4. Evolve the initial state for the same time range and time step used in the previous exercizes.


In [ ]:
# Obtain the same amplitudes from the MPS 

nn = 20
jj=1.0
system = siteinds("S=1/2",nn)
InitialUps = [1, 2, 3, 4, 5, 20, 19, 18, 17, 16]
psi0= MPS(ComplexF64,system,[j in InitialUps ? "Up" : "Dn" for j in 1:nn]);
#Set the orthogonality center to the first site
orthogonalize!(psi0,1)

In [ ]:
opsum = OpSum()
for bond in 1:nn-1
    #note that the sintax allows for constants to appear before the operator string
    opsum += -1. * jj, "S-", bond, "S+", bond+1
    opsum += -1. * jj, "S+", bond, "S-", bond+1
end
ham = MPO(opsum, system)

In [ ]:
deltat = 0.01
nsteps = Int(round(10/deltat))
hdeltat = deltat * ham
#build an MPO with the operator "Id" on each site
theOne = MPO(system,"Id")
factor(k) = (-1im)^k / factorial(k)
#As not to have all the indices of hdeltat contracted with each other, we prime the second one and then we contract the first index of the second one with the first index of the first one, and then we lower the second prime level to first prime level.
h2 = mapprime(prime(hdeltat)*hdeltat,2 =>1)
h3 = mapprime(prime(hdeltat) * h2,2 =>1)
terms = [theOne, factor(1)*hdeltat, factor(2)*h2, factor(3)*h3];
appo = terms[1];
for t in terms[2:end]
    appo = t + appo
end
h4thOrder = appo;
#h4thOrder = +(terms...; cutoff=1e-18);

In [ ]:
evolved = Vector{MPS}()
push!(evolved, psi0)
for j=2:nsteps
    push!(evolved, normalize(noprime(*(h4thOrder, evolved[end]; cutoff=1e-3, maxdim=50))))
    if j % 100 == 0
        println("Step $j / $nsteps done")
    end
end

evolved[1]